In [ ]:
# =============================================================================
# CELL 1: LIGHTWEIGHT SETUP WITH OPTIMIZED KEEP-ALIVE
# =============================================================================

# Quick GPU check and install
!nvidia-smi | head -10
!pip install librosa tensorflow pandas numpy matplotlib scikit-learn -q
!pip install tqdm joblib soundfile -q

# Lightweight keep-alive (reduced frequency)
from IPython.display import Javascript, display
import threading
import time

display(Javascript('''
window.keepAlive = function() {
    const btn = document.querySelector('colab-connect-button');
    if (btn) btn.click();
    document.dispatchEvent(new Event('mousemove'));
    setTimeout(window.keepAlive, 60000); // Every 1 minute
};
window.keepAlive();
console.log("Keep-alive started");
'''))

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {len(tf.config.list_physical_devices('GPU'))} devices")


Tue Aug 26 13:59:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |


<IPython.core.display.Javascript object>

TensorFlow: 2.19.0
GPU: 1 devices


In [ ]:
# =============================================================================
# CELL 2: MOUNT DRIVE AND VERIFY PATHS
# =============================================================================

from google.colab import drive
import os
import pandas as pd

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/TotalFiles'
TRAIN_CSV = f'{BASE_PATH}/train.csv'
TRAIN_AUDIO_DIR = f'{BASE_PATH}/train'
VALID_CSV = f'{BASE_PATH}/valid.csv'
VALID_AUDIO_DIR = f'{BASE_PATH}/valid'
TEST_CSV = f'{BASE_PATH}/test.csv'
TEST_AUDIO_DIR = f'{BASE_PATH}/test'

# Quick verification
for name, path in [('Train CSV', TRAIN_CSV), ('Train Audio', TRAIN_AUDIO_DIR)]:
    if os.path.exists(path):
        if path.endswith('.csv'):
            count = len(pd.read_csv(path))
        else:
            count = len([f for f in os.listdir(path) if f.endswith('.wav')])
        print(f"✓ {name}: {count} files")
    else:
        print(f"✗ {name}: Not found")



Mounted at /content/drive
✓ Train CSV: 2041 files
✓ Train Audio: 2041 files


In [ ]:
# =============================================================================
# CELL 3: ULTRA-LIGHTWEIGHT CONFIGURATION
# =============================================================================

import numpy as np
import librosa
import gc
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

class OptimizedConfig:
    # Drastically reduced parameters for T4 GPU
    SAMPLE_RATE = 16000
    CHUNK_DURATION = 0.05  # 50ms chunks (larger = fewer chunks)
    CHUNK_SAMPLES = 800    # 50ms at 16kHz

    # Minimal features for speed
    N_MFCC = 8            # Reduced from 13
    FEATURE_DIM = 12      # Total features

    # Aggressive model optimization
    SEQUENCE_LENGTH = 20   # Reduced from 50
    BATCH_SIZE = 16       # Smaller batches
    EPOCHS = 8            # Fewer epochs
    LEARNING_RATE = 0.003

    # Resource limits
    MAX_FILES = 150       # Drastically reduced dataset
    MAX_CHUNKS_PER_FILE = 200  # Limit chunks per file
    MAX_AUDIO_DURATION = 15    # Max 15 seconds per file

    # Labels
    RINGTONE_LABEL = 0
    CLEAN_SPEECH_LABEL = 1

config = OptimizedConfig()
print(f"Config: {config.MAX_FILES} files, {config.FEATURE_DIM}D features, {config.SEQUENCE_LENGTH} seq")


Config: 150 files, 12D features, 20 seq


In [ ]:
# =============================================================================
# CELL 4: CPU-OPTIMIZED FEATURE EXTRACTION
# =============================================================================

def extract_minimal_features(audio, sr=config.SAMPLE_RATE):
    """CPU-optimized feature extraction - addresses main bottleneck"""
    try:
        # Ensure correct length (minimal padding/truncation)
        if len(audio) != config.CHUNK_SAMPLES:
            if len(audio) < config.CHUNK_SAMPLES:
                audio = np.pad(audio, (0, config.CHUNK_SAMPLES - len(audio)))
            else:
                audio = audio[:config.CHUNK_SAMPLES]

        features = []

        # CRITICAL: Simplified MFCC with fewer computations
        try:
            # Reduced FFT size and hop length for speed
            mfcc = librosa.feature.mfcc(
                y=audio, sr=sr, n_mfcc=config.N_MFCC,
                hop_length=400,  # Larger hop = fewer computations
                n_fft=512        # Smaller FFT = faster
            )
            features.extend(np.mean(mfcc, axis=1))
        except:
            # Fast fallback - no librosa
            fft = np.fft.fft(audio)
            magnitude = np.abs(fft[:config.N_MFCC])
            features.extend(magnitude / len(audio))

        # Ultra-fast statistical features (CPU-friendly)
        features.extend([
            np.mean(audio),           # Mean
            np.std(audio),            # Std
            np.max(np.abs(audio)),    # Peak
            np.sum(audio**2)/len(audio)  # RMS energy
        ])

        return np.array(features[:config.FEATURE_DIM])

    except:
        # Emergency fallback - pure NumPy (no librosa)
        return np.array([
            np.mean(audio), np.std(audio), np.max(audio), np.min(audio),
            np.median(audio), np.var(audio), np.sum(audio**2), 0, 0, 0, 0, 0
        ][:config.FEATURE_DIM])

def process_single_file(audio_path, csv_row):
    """Process single audio file with aggressive optimization"""
    try:
        if not os.path.exists(audio_path):
            return None, None

        # Load with duration limit
        audio, sr = librosa.load(audio_path, sr=config.SAMPLE_RATE,
                                duration=config.MAX_AUDIO_DURATION, mono=True)

        if len(audio) < config.CHUNK_SAMPLES:
            return None, None

        # Calculate chunks with limit
        n_chunks = min(len(audio) // config.CHUNK_SAMPLES, config.MAX_CHUNKS_PER_FILE)

        chunks = []
        labels = []

        # Get ringtone timing
        try:
            ringtone_start = float(csv_row.get('ringtone_start', 0))
            ringtone_end = float(csv_row.get('ringtone_end', 0))
        except:
            ringtone_start = ringtone_end = 0

        for i in range(n_chunks):
            start_sample = i * config.CHUNK_SAMPLES
            chunk = audio[start_sample:start_sample + config.CHUNK_SAMPLES]

            # Extract features
            features = extract_minimal_features(chunk)
            chunks.append(features)

            # Determine label
            chunk_time = (start_sample + config.CHUNK_SAMPLES//2) / sr
            if ringtone_start <= chunk_time <= ringtone_end:
                labels.append(config.RINGTONE_LABEL)
            else:
                labels.append(config.CLEAN_SPEECH_LABEL)

        return np.array(chunks), np.array(labels)

    except:
        return None, None

print("Minimal feature extraction ready")


Minimal feature extraction ready


In [ ]:
# =============================================================================
# CELL 5: AGGRESSIVE DATA LOADING
# =============================================================================

def load_dataset_aggressively(csv_path, audio_dir, max_files=config.MAX_FILES):
    """Load dataset with aggressive optimization"""
    print(f"Loading max {max_files} files from {audio_dir}")

    df = pd.read_csv(csv_path)

    # Random sample of files
    if len(df) > max_files:
        df = df.sample(n=max_files, random_state=42).reset_index(drop=True)

    all_features = []
    all_labels = []
    successful_files = 0

    # Get WAV files
    wav_files = [f for f in os.listdir(audio_dir) if f.endswith('.wav')]

    print(f"Processing {len(df)} files...")

    for idx in tqdm(range(len(df)), desc="Loading"):
        # Simple file matching - take by index
        if idx < len(wav_files):
            audio_path = os.path.join(audio_dir, wav_files[idx])
            row = df.iloc[idx] if idx < len(df) else df.iloc[0]

            features, labels = process_single_file(audio_path, row)

            if features is not None and len(features) > 0:
                all_features.append(features)
                all_labels.append(labels)
                successful_files += 1

            # Memory cleanup every 20 files
            if successful_files % 20 == 0:
                gc.collect()

    if not all_features:
        return None, None

    # Concatenate all data
    final_features = np.concatenate(all_features, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)

    print(f"✓ Loaded {successful_files} files")
    print(f"✓ Features shape: {final_features.shape}")
    print(f"✓ Ringtone ratio: {np.mean(final_labels == 0):.2%}")

    return final_features, final_labels

# Test loading
print("Testing data loading...")
test_features, test_labels = load_dataset_aggressively(TRAIN_CSV, TRAIN_AUDIO_DIR, max_files=5)
if test_features is not None:
    print(f"✓ Test successful: {test_features.shape}")
else:
    print("✗ Test failed")


Testing data loading...
Loading max 5 files from /content/drive/MyDrive/TotalFiles/train
Processing 5 files...


Loading: 100%|██████████| 5/5 [00:27<00:00,  5.43s/it]

✓ Loaded 5 files
✓ Features shape: (1000, 12)
✓ Ringtone ratio: 0.00%
✓ Test successful: (1000, 12)


In [ ]:
# =============================================================================
# CELL 6: FIXED COMPACT LSTM MODEL
# =============================================================================

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, TimeDistributed
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

def create_compact_model():
    """Ultra-compact LSTM model for T4 GPU - FIXED SHAPE ISSUE"""
    model = Sequential([
        Input(shape=(config.SEQUENCE_LENGTH, config.FEATURE_DIM)),

        # Single LSTM layer - minimal complexity
        LSTM(16, return_sequences=True, dropout=0.2),

        # TimeDistributed layers to match sequence output shape
        TimeDistributed(Dense(8, activation='relu')),
        TimeDistributed(Dropout(0.3)),
        TimeDistributed(Dense(1, activation='sigmoid'))  # Single output per timestep
    ])

    model.compile(
        optimizer=Adam(learning_rate=config.LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# Create and test model
model = create_compact_model()
model.summary()

total_params = model.count_params()
print(f"Total parameters: {total_params:,}")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 20, 16)         │         1,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 20, 8)          │           136 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 8)          │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 20, 1)          │             9 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,001 (7.82 KB)

 Trainable params: 2,001 (7.82 KB)

 Non-trainable params: 0 (0.00 B)

Total parameters: 2,001


In [ ]:
# =============================================================================
# CELL 7: SEQUENCE CREATION AND NORMALIZATION
# =============================================================================

def create_sequences(features, labels, seq_len=config.SEQUENCE_LENGTH):
    """Create LSTM sequences efficiently"""
    if len(features) < seq_len:
        return None, None

    sequences = []
    sequence_labels = []

    # Take every other sequence to reduce memory
    step = 2
    for i in range(0, len(features) - seq_len + 1, step):
        sequences.append(features[i:i + seq_len])
        sequence_labels.append(labels[i:i + seq_len])

    return np.array(sequences), np.array(sequence_labels)

def normalize_features(features):
    """Normalize features efficiently"""
    scaler = StandardScaler()
    original_shape = features.shape
    features_flat = features.reshape(-1, features.shape[-1])
    features_normalized = scaler.fit_transform(features_flat)
    features_normalized = features_normalized.reshape(original_shape)
    return features_normalized, scaler

print("Sequence functions ready")

Sequence functions ready


In [ ]:
# =============================================================================
# CELL 8: OPTIMIZED TRAINING PIPELINE
# =============================================================================

def train_optimized():
    """Ultra-optimized training pipeline"""
    print("STARTING OPTIMIZED TRAINING")
    print("=" * 40)

    # Load training data
    print("1. Loading training data...")
    train_features, train_labels = load_dataset_aggressively(
        TRAIN_CSV, TRAIN_AUDIO_DIR, config.MAX_FILES
    )

    if train_features is None:
        print("✗ Failed to load training data")
        return None, None, None

    # Create sequences
    print("2. Creating sequences...")
    train_sequences, train_seq_labels = create_sequences(train_features, train_labels)

    if train_sequences is None:
        print("✗ Failed to create sequences")
        return None, None, None

    print(f"✓ Training sequences: {train_sequences.shape}")

    # Normalize
    print("3. Normalizing features...")
    train_normalized, scaler = normalize_features(train_sequences)

    # Load validation data (smaller subset)
    print("4. Loading validation data...")
    try:
        valid_features, valid_labels = load_dataset_aggressively(
            VALID_CSV, VALID_AUDIO_DIR, max_files=30
        )

        if valid_features is not None:
            valid_sequences, valid_seq_labels = create_sequences(valid_features, valid_labels)
            if valid_sequences is not None:
                valid_normalized = scaler.transform(
                    valid_sequences.reshape(-1, valid_sequences.shape[-1])
                ).reshape(valid_sequences.shape)
                print(f"✓ Validation sequences: {valid_normalized.shape}")
            else:
                valid_normalized = valid_seq_labels = None
        else:
            valid_normalized = valid_seq_labels = None
    except:
        valid_normalized = valid_seq_labels = None
        print("⚠ No validation data")

    # Setup callbacks
    callbacks = [
        EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
        ModelCheckpoint('compact_model.h5', save_best_only=True, verbose=1),
        ReduceLROnPlateau(patience=2, factor=0.5, verbose=1)
    ]

    # Train model
    print("5. Training model...")
    print(f"   Epochs: {config.EPOCHS}")
    print(f"   Batch size: {config.BATCH_SIZE}")

    validation_data = None
    if valid_normalized is not None:
        validation_data = (valid_normalized, valid_seq_labels)

    history = model.fit(
        train_normalized, train_seq_labels,
        batch_size=config.BATCH_SIZE,
        epochs=config.EPOCHS,
        validation_data=validation_data,
        callbacks=callbacks,
        verbose=1
    )

    # Save scaler
    import joblib
    joblib.dump(scaler, 'compact_scaler.pkl')

    print("✓ TRAINING COMPLETED")
    print("✓ Model saved: compact_model.h5")
    print("✓ Scaler saved: compact_scaler.pkl")

    return model, history, scaler

# Execute training
model, history, scaler = train_optimized()


STARTING OPTIMIZED TRAINING
1. Loading training data...
Loading max 150 files from /content/drive/MyDrive/TotalFiles/train
Processing 150 files...


Loading: 100%|██████████| 150/150 [07:43<00:00,  3.09s/it]


✓ Loaded 150 files
✓ Features shape: (30000, 12)
✓ Ringtone ratio: 0.00%
2. Creating sequences...
✓ Training sequences: (14991, 20, 12)
3. Normalizing features...
4. Loading validation data...
Loading max 30 files from /content/drive/MyDrive/TotalFiles/valid
Processing 30 files...


Loading: 100%|██████████| 30/30 [00:32<00:00,  1.09s/it]

✓ Loaded 30 files
✓ Features shape: (6000, 12)
✓ Ringtone ratio: 0.00%
✓ Validation sequences: (2991, 20, 12)
5. Training model...
   Epochs: 8
   Batch size: 16
Epoch 1/8


...

937/937 ━━━━━━━━━━━━━━━━━━━━ 21s 15ms/step - accuracy: 1.0000 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 8.2564e-07 - learning_rate: 0.0015
Epoch 5/8
935/937 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.0011
Epoch 5: val_loss improved from 0.00000 to 0.00000, saving model to compact_model.h5



Epoch 5: ReduceLROnPlateau reducing learning rate to 0.000750000006519258.
937/937 ━━━━━━━━━━━━━━━━━━━━ 20s 15ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 4.1723e-07 - learning_rate: 0.0015
Epoch 6/8
934/937 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 9.0703e-04
Epoch 6: val_loss improved from 0.00000 to 0.00000, saving model to compact_model.h5


937/937 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 1.0000 - loss: 9.0702e-04 - val_accuracy: 1.0000 - val_loss: 2.8628e-07 - learning_rate: 7.5000e-04
Epoch 7/8
934/937 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 9.1752e-04
Epoch 7: val_loss improved from 0.00000 to 0.00000, saving model to compact_model.h5



Epoch 7: ReduceLROnPlateau reducing learning rate to 0.000375000003259629.
937/937 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - accuracy: 1.0000 - loss: 9.1707e-04 - val_accuracy: 1.0000 - val_loss: 1.9689e-07 - learning_rate: 7.5000e-04
Epoch 8/8
936/937 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 8.0659e-04
Epoch 8: val_loss improved from 0.00000 to 0.00000, saving model to compact_model.h5


937/937 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - accuracy: 1.0000 - loss: 8.0644e-04 - val_accuracy: 1.0000 - val_loss: 1.6778e-07 - learning_rate: 3.7500e-04
Restoring model weights from the end of the best epoch: 8.
✓ TRAINING COMPLETED
✓ Model saved: compact_model.h5
✓ Scaler saved: compact_scaler.pkl
